# 03 — Jordan: Region Detection & IoU

**Google Colab notebook.** Runtime → *Change runtime type* → **GPU (T4 / L4 / A100)** before running.

Trains a region detector on the **OCR Dataset of Multi-type Documents** — which ships **52,331 real polygon boxes with transcriptions**. No synthetic or heuristic boxes are needed.

| | |
|---|---|
| **Inputs** | Drive: `datasets/ocr_multitype/`, invoice manifest |
| **Outputs** | `region_predictions.csv`, `region_iou_metrics.json`, figure, weights |
| **Expected runtime** | ~45–75 min on a T4 |
| **Compute profile** | `colab_gpu` (generous — full data, pinned in the profile cell) |

### How results get back to the team
Everything is written to Google Drive by `colab_bootstrap.publish()`, into **both**:
- `outputs/jordan/<kind>/` — the *latest* copy
- `runs/jordan/<UTC-timestamp>/<kind>/` — an immutable archive, so re-running never
  silently destroys an earlier result

Tell the integrator (Hessam) when you're done; he copies from `outputs/` into the repo.

> **Before you run:** `MyDrive/DL2_InvoiceAI/` must already contain `code/` (the repo's `src/`,
> `scripts/`, and `colab_bootstrap.py`) and `inputs/`. If it doesn't, the bootstrap cell fails
> fast with a message telling you exactly what's missing.

### Read this — your brief changed

An earlier plan said *"region bboxes are NOT in the annotation CSVs, so use a heuristic/synthetic
region-box approach."* **That is obsolete.** The `OCR Dataset of Multi-type Documents` was found
sitting unused in the raw data and it contains exactly what this stage needs:

| | |
|---|---|
| Images | 973, pre-split **778 / 97 / 98** |
| Annotation pairing | **100%** |
| **Polygon boxes + text** | **52,331** (median 50/image) |
| Entity fields | `company`, `date`, `address`, `total` |

```json
{"file_id": "X00016469612",
 "entities": {"company": "...", "date": "...", "address": "...", "total": "9.00"},
 "ocr_boxes": [{"points": [[72,25],[326,25],[326,64],[72,64]], "text": "TAN WOON YANN"}]}
```

### How unlabeled text boxes become *labeled regions*
`ocr_boxes` are text lines with no class. `entities` give field **values** but no coordinates.
We join them: **match entity text against box text** to label those boxes `company` / `date` /
`address` / `total`, and everything else becomes `other_text`. That yields a real 5-class region
detection problem with real geometry — and it feeds Damir directly.

### Domain gap
These are **receipts** (~460 px wide); the invoice corpus is **full-page** (1654×2339). A detector
trained here will not transfer perfectly. Metrics come from the dataset's own test split; invoice
inference is reported as counts.

In [ ]:
# --- GPU check: stop here if this says "no GPU" ---------------------------------
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "!! NO GPU. Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# --- Mount Drive + load the shared bootstrap -----------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs

import sys, os, shutil, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), (
    f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
    f"{DRIVE_ROOT}/code/ and re-run this cell."
)
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("ultralytics", "opencv-python-headless", "pandas", "rapidfuzz")
print("Drive root:", root)

In [ ]:
# --- Pin the generous Colab budget --------------------------------------------
os.environ["IIP_COMPUTE_PROFILE"] = "colab_gpu"
from src.compute_profile import get_profile

P = get_profile()
print(json.dumps(P, indent=2, default=str))

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())   # one archive folder for this run
T0 = time.time()

In [ ]:
# --- Datasets: read straight from Google Drive (NO Kaggle token needed) ------
# 548 MB, pre-downloaded into Drive. This is the dataset with the 52,331 real polygon boxes.
# Paths are resolved tolerantly: if a dataset was copied one level too deep
# (e.g. ocr_multitype/invoice/train/... instead of ocr_multitype/train/...),
# it is found anyway and a NOTE is printed. No re-upload needed.
DATA = paths.inputs / "datasets"

BASE = CB.resolve_dataset_root(DATA / "ocr_multitype", ['train/annotations', 'val/annotations', 'test/annotations'])

print(f"  ocr_multitype  -> {BASE}")
print(f"                    " f"{sum(1 for _ in BASE.rglob(chr(42)) if _.is_file()):,} files")

In [ ]:
# --- Parse the JSON annotations ----------------------------------------------
import pandas as pd, numpy as np, cv2
from rapidfuzz import fuzz

print("dataset root:", BASE)
for sp in ["train", "val", "test"]:
    print(f"  {sp}: {len(list((BASE/sp/'annotations').glob('*.json')))} ann, "
          f"{len(list((BASE/sp/'images').glob('*')))} imgs")

def load(sp):
    out = []
    for ap in sorted((BASE / sp / "annotations").glob("*.json")):
        d = json.loads(ap.read_text(encoding="utf-8"))
        img = next((BASE / sp / "images").glob(ap.stem + ".*"), None)
        if img is None:
            continue
        out.append({"split": sp, "file_id": d.get("file_id", ap.stem), "img": img,
                    "entities": d.get("entities", {}), "boxes": d.get("ocr_boxes", [])})
    return out

data = {sp: load(sp) for sp in ["train", "val", "test"]}
tot = sum(len(v) for v in data.values())
nbox = sum(len(r["boxes"]) for v in data.values() for r in v)
print(f"\nloaded {tot} documents, {nbox} boxes")

In [ ]:
# --- Label boxes by matching entity text -------------------------------------
FIELDS = ["company", "date", "address", "total"]
CLASSES = FIELDS + ["other_text"]          # index order is fixed - do not reorder

def norm(s):
    return " ".join(str(s).upper().split())

def quad_to_xyxy(points):
    a = np.asarray(points, dtype=float)
    return [float(a[:, 0].min()), float(a[:, 1].min()),
            float(a[:, 0].max()), float(a[:, 1].max())]

def label_boxes(rec, thresh=88):
    ents = {k: norm(v) for k, v in rec["entities"].items() if v}
    out = []
    for b in rec["boxes"]:
        t = norm(b.get("text", ""))
        cls, best = 4, 0                       # default other_text
        if t:
            for i, f in enumerate(FIELDS):
                e = ents.get(f)
                if not e:
                    continue
                s = max(fuzz.partial_ratio(t, e), fuzz.ratio(t, e))
                if s > best and s >= thresh:
                    best, cls = s, i
        out.append((cls, quad_to_xyxy(b["points"])))
    return out

dist = {c: 0 for c in CLASSES}
for r in data["train"]:
    for ci, _ in label_boxes(r):
        dist[CLASSES[ci]] += 1
print("train box class distribution:")
for k, v in dist.items():
    print(f"  {k:12s} {v:7d}")
print("\nNOTE the imbalance: other_text dominates. Report per-class metrics, not just mAP.")

In [ ]:
# --- Build the YOLO dataset (use the dataset's own splits) -------------------
D = Path("/content/yolo_regions")
for sp in ["train", "val", "test"]:
    (D / sp / "images").mkdir(parents=True, exist_ok=True)
    (D / sp / "labels").mkdir(parents=True, exist_ok=True)

cap = P.get("max_images_per_class") or 10**9
counts = {}
for sp, recs in data.items():
    n = 0
    for rec in recs[:cap]:
        im = cv2.imread(str(rec["img"]))
        if im is None:
            continue
        H, W = im.shape[:2]
        dst = D / sp / "images" / f"{rec['file_id']}.jpg"
        cv2.imwrite(str(dst), im)
        lines = []
        for ci, (x1, y1, x2, y2) in label_boxes(rec):
            cx, cy = (x1+x2)/2/W, (y1+y2)/2/H
            bw, bh = (x2-x1)/W, (y2-y1)/H
            if bw <= 0 or bh <= 0:
                continue
            lines.append(f"{ci} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        (D / sp / "labels" / f"{rec['file_id']}.txt").write_text("\n".join(lines))
        n += 1
    counts[sp] = n

yaml = D / "data.yaml"
yaml.write_text(f"path: {D}\ntrain: train/images\nval: val/images\ntest: test/images\n"
                f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")
print(counts)

In [ ]:
# --- Train --------------------------------------------------------------------
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data=str(yaml), epochs=P["epochs"], imgsz=P["imgsz"], batch=P["batch"],
            workers=P.get("workers", 2), device=0, patience=P.get("patience", 20),
            project="/content/runs", name="regions", exist_ok=True, verbose=True)
BEST = Path("/content/runs/regions/weights/best.pt")
print("best:", BEST.exists())

In [ ]:
# --- Per-class IoU on the dataset's real TEST split ---------------------------
from src.iou import compute_iou
m = YOLO(str(BEST)); CONF = 0.25

stats = {c: {"tp": 0, "fp": 0, "fn": 0, "ious": []} for c in CLASSES}
for ip in sorted((D/"test"/"images").glob("*.jpg")):
    im = cv2.imread(str(ip)); H, W = im.shape[:2]
    gt = []
    for ln in (D/"test"/"labels"/f"{ip.stem}.txt").read_text().splitlines():
        if not ln.strip():
            continue
        ci, cx, cy, bw, bh = ln.split(); ci = int(ci)
        cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        gt.append((ci, [(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H]))
    pr = m.predict(str(ip), conf=CONF, verbose=False)[0]
    preds = [(int(c), b.tolist()) for c, b in
             zip(pr.boxes.cls.cpu().numpy(), pr.boxes.xyxy.cpu().numpy())]
    for ci, name in enumerate(CLASSES):
        g = [b for k, b in gt if k == ci]
        p_ = [b for k, b in preds if k == ci]
        used = set()
        for pb in p_:
            best, bi = 0.0, -1
            for j, gb in enumerate(g):
                if j in used:
                    continue
                v = compute_iou(pb, gb)
                if v > best:
                    best, bi = v, j
            if best >= 0.5:
                stats[name]["tp"] += 1; stats[name]["ious"].append(best); used.add(bi)
            else:
                stats[name]["fp"] += 1
        stats[name]["fn"] += len(g) - len(used)

region_metrics = {}
for n_, s in stats.items():
    tp, fp, fn = s["tp"], s["fp"], s["fn"]
    region_metrics[n_] = {
        "precision": round(tp/(tp+fp), 4) if tp+fp else 0.0,
        "recall":    round(tp/(tp+fn), 4) if tp+fn else 0.0,
        "mean_iou":  round(float(np.mean(s["ious"])), 4) if s["ious"] else 0.0,
        "support":   tp+fn,
    }
print(json.dumps(region_metrics, indent=2))

In [ ]:
# --- Provenance: the _run block makes cross-run model comparison possible ------
def run_block(**kw):
    """Stamp every metrics JSON with how it was produced, so local-CPU and Colab-GPU
    results can be charted against each other later."""
    b = {
        "profile": P.get("profile_name", "colab_gpu"),
        "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
        "epochs": P.get("epochs"), "imgsz": P.get("imgsz"), "batch": P.get("batch"),
        "wall_clock_sec": round(time.time() - T0, 1),
        "timestamp_utc": RUN_TS, "member": "jordan",
    }
    b.update(kw)
    return b


In [ ]:
# --- Predictions on the dataset test split + on real invoices ----------------
OUTD = Path("/content/out"); OUTD.mkdir(exist_ok=True)
rows = []
for ip in sorted((D/"test"/"images").glob("*.jpg")):
    pr = m.predict(str(ip), conf=CONF, verbose=False)[0]
    for cls, box, cf in zip(pr.boxes.cls.cpu().numpy(), pr.boxes.xyxy.cpu().numpy(),
                            pr.boxes.conf.cpu().numpy()):
        rows.append({"document_id": ip.stem, "image_path": f"ocrset/test/{ip.name}",
                     "region_label": CLASSES[int(cls)],
                     "xmin": float(box[0]), "ymin": float(box[1]),
                     "xmax": float(box[2]), "ymax": float(box[3]),
                     "confidence": round(float(cf), 4), "source": "ocr_dataset_test"})

man = pd.read_csv(paths.inputs / "invoice_manifest.csv")
for r in man.itertuples():
    ip = root / r.image_path
    if not ip.exists():
        continue
    pr = m.predict(str(ip), conf=CONF, verbose=False)[0]
    for cls, box, cf in zip(pr.boxes.cls.cpu().numpy(), pr.boxes.xyxy.cpu().numpy(),
                            pr.boxes.conf.cpu().numpy()):
        rows.append({"document_id": r.document_id, "image_path": r.image_path,
                     "region_label": CLASSES[int(cls)],
                     "xmin": float(box[0]), "ymin": float(box[1]),
                     "xmax": float(box[2]), "ymax": float(box[3]),
                     "confidence": round(float(cf), 4), "source": "invoice"})

reg = pd.DataFrame(rows)
reg.to_csv(OUTD / "region_predictions.csv", index=False)
print(reg.groupby(["source", "region_label"]).size())

In [ ]:
# --- Figure + metrics ---------------------------------------------------------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

FIG = Path("/content/out/figures"); FIG.mkdir(parents=True, exist_ok=True)
COL = {"company": "tab:red", "date": "tab:green", "address": "tab:orange",
       "total": "tab:purple", "other_text": "tab:gray"}
te = reg[reg.source == "ocr_dataset_test"]
fig, ax = plt.subplots(1, 4, figsize=(16, 7))
for a, d in zip(ax.ravel(), te.document_id.drop_duplicates().head(4)):
    a.imshow(plt.imread(D/"test"/"images"/f"{d}.jpg")); a.axis("off"); a.set_title(d, fontsize=8)
    for q in te[te.document_id == d].itertuples():
        a.add_patch(Rectangle((q.xmin, q.ymin), q.xmax-q.xmin, q.ymax-q.ymin,
                              fill=False, lw=1.4, ec=COL.get(q.region_label, "k")))
fig.suptitle("Region detection on the OCR Dataset test split "
             "(red=company, green=date, orange=address, purple=total, gray=other)", fontsize=11)
fig.tight_layout(); fig.savefig(FIG/"region_detection_examples.png", dpi=150); plt.close(fig)

MD = Path("/content/out/models/region_detector"); MD.mkdir(parents=True, exist_ok=True)
shutil.copyfile(BEST, MD/"best.pt")
(MD/"README.md").write_text(
    f"# region_detector\n\nYOLOv8n, {len(CLASSES)} classes: {CLASSES}\n\n"
    f"Trained on the OCR Dataset of Multi-type Documents (real polygon boxes; entity text "
    f"matched to box text to assign field labels).\nProfile colab_gpu: epochs={P['epochs']}, "
    f"imgsz={P['imgsz']}.\n\nMetrics: {json.dumps(region_metrics)}\n", encoding="utf-8")

met = Path("/content/out/region_iou_metrics.json")
payload = {"per_class": region_metrics,
           "macro_mean_iou": round(float(np.mean([v["mean_iou"] for v in region_metrics.values()])), 4),
           "_run": run_block(model="yolov8n", classes=CLASSES,
                             n_train_images=counts.get("train"),
                             eval_set="OCR Dataset official test split (98 imgs)",
                             conf_threshold=CONF, iou_match_threshold=0.5),
           "_invoice_inference": {
               "note": "Receipts -> full-page invoices is a domain shift; counts only, no GT.",
               "invoices_with_regions": int(reg[reg.source == 'invoice'].document_id.nunique()),
           }}
met.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps(payload, indent=2)[:900])

In [ ]:
# --- Publish to Drive (latest + immutable archive) -----------------------------
# 
to_publish = [
    ("predictions", OUTD / "region_predictions.csv"),
    ("metrics", met),
    ("figures", FIG / "region_detection_examples.png"),
    ("models", Path("/content/out/models")),
]
for kind, src in to_publish:
    if src is None:
        continue
    p = Path(src)
    if not p.exists():
        print(f"  skip (not produced): {p}")
        continue
    CB.publish("jordan", p, kind, paths=paths, run_timestamp=RUN_TS)

print("\nLatest ->", paths.outputs("jordan"))
print("Archive ->", paths.run_dir("jordan", timestamp=RUN_TS))

In [ ]:
# --- Hand off to Damir + Hessam ----------------------------------------------
up = paths.inputs / "upstream" / "jordan"; up.mkdir(parents=True, exist_ok=True)
shutil.copyfile(OUTD/"region_predictions.csv", up/"region_predictions.csv")
print("handed off ->", up)

## Report log — fill this in before you finish

Copy your answers into `presentation/member_reports/jordan_report_log.md` in the repo (or paste
them to the integrator). This is the raw material for the group report and slide deck, so be
specific and **honest about what didn't work**.

1. The entity-text → box-text matching trick: what fuzzy threshold, and what did it mislabel?
2. Class imbalance — `other_text` dwarfs the four field classes. What did you do about it?
3. Per-class precision/recall/IoU on the real test split; which field is hardest and why.
4. The receipt → full-page-invoice domain gap: what did predictions look like on invoices?
5. How this compares to the heuristic-box approach originally planned (which you avoided).

Also note anything the next stage needs from you, and which figure you'd put on a slide.